# **Minimax-H3 for AI Video Generation (ComfyUI)**
- Run the cell below to get a link (e.g. https://literature-consortium-align.trycloudflare.com ) which you can use to launch the comfyUI interface.
- If you get an error on the resize image node, change the upscale_method.
* You can get models and workflows here:
  * (1) https://huggingface.co/Abiray/MiniMax-H3-GGUF/tree/main  
  * (2) https://huggingface.co/realrebelai/MiniMax-H3_GGUFs/tree/main
- Huggingface page: https://huggingface.co/MiniMaxAI/MiniMax-H3
- Notebook source: https://github.com/Isi-dev/Google-Colab_Notebooks

In [ ]:
# @title # 1. 💥 Prepare Environment & Install Dependencies {"single-column":true}
import os

# 1. ALWAYS force the directory back to the Colab root before cloning
%cd /content

# 2. Clone ComfyUI only if it doesn't already exist in the root
if not os.path.exists("ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git

# 3. Move into the correct, top-level ComfyUI folder
%cd /content/ComfyUI

# Install core dependencies
!pip install -r requirements.txt
from IPython.display import clear_output

# 4. Install custom nodes only if they don't already exist
if not os.path.exists("custom_nodes/ComfyUI-GGUF"):
    !cd custom_nodes && git clone https://github.com/city96/ComfyUI-GGUF.git

# 5. Install Video Helper Suite
if not os.path.exists("custom_nodes/ComfyUI-VideoHelperSuite"):
    !cd custom_nodes && git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
    !cd custom_nodes/ComfyUI-VideoHelperSuite && pip install -r requirements.txt

# 6. Install KJNodes
if not os.path.exists("custom_nodes/ComfyUI-KJNodes"):
    !cd custom_nodes && git clone https://github.com/kijai/ComfyUI-KJNodes.git
    !cd custom_nodes/ComfyUI-KJNodes && pip install -r requirements.txt

# Install GGUF and high-speed Hugging Face download engine
!pip install gguf huggingface_hub hf_transfer

clear_output()


# @markdown Select your desired models below.

# @markdown ---
# @markdown ### **HuggingFace Token (Optional but Recommended)**
# @markdown Adding a free token prevents rate-limiting and stalling on massive downloads. Get yours at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
hf_token = "" # @param {type:"string"}

# @markdown ---
# @markdown ### **Text Encoders**
text_encoder_model = "qwen3vl_32b_minimax_h3-Q4_K_M.gguf" # @param ["qwen3vl_32b_minimax_h3-Q4_K_M.gguf", "qwen3vl_32b_minimax_h3_int4_convrot.safetensors", "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "None"]

download_audio_vae = True
download_video_vae = True

# @markdown ---
# @markdown ### **UNet Models (GGUF)**
unet_model = "MiniMax-H3-FL2VA-Q4_K_M.gguf" # @param ["MiniMax-H3-FL2VA-Q3_K_M.gguf", "MiniMax-H3-FL2VA-Q4_0.gguf", "MiniMax-H3-FL2VA-Q4_K_M.gguf", "MiniMax-H3-FL2VA-Q4_K_S.gguf", "MiniMax-H3-FL2VA-Q5_0.gguf", "MiniMax-H3-FL2VA-Q5_K_M.gguf", "MiniMax-H3-FL2VA-Q5_K_S.gguf", "MiniMax-H3-FL2VA-Q6_K.gguf", "MiniMax-H3-FL2VA-Q8_0.gguf", "MiniMax-H3-Ref2VA-Q3_K_M.gguf", "MiniMax-H3-Ref2VA-Q4_0.gguf", "MiniMax-H3-Ref2VA-Q4_K_M.gguf", "MiniMax-H3-Ref2VA-Q4_K_S.gguf", "MiniMax-H3-Ref2VA-Q5_0.gguf", "MiniMax-H3-Ref2VA-Q5_K_M.gguf", "MiniMax-H3-Ref2VA-Q5_K_S.gguf", "None"]

import os
from pathlib import Path
from huggingface_hub import hf_hub_download

# Ensure environment is set to ComfyUI root directory
%cd /content/ComfyUI

# Enable Rust-accelerated fast transfers
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

REPO_ID = "Abiray/MiniMax-H3-GGUF"

def download_file(repo_relative_path, token):
    if not repo_relative_path or "None" in repo_relative_path:
        return

    # Target path inside ComfyUI/models/
    destination_file = os.path.join("models", repo_relative_path)
    filename_only = os.path.basename(repo_relative_path)

    if os.path.exists(destination_file):
        print(f"⏭️ Skipping {filename_only} (Already exists in models/)\n")
        return

    print(f"Downloading {filename_only} via hf_transfer...")
    try:
        hf_hub_download(
            repo_id=REPO_ID,
            filename=repo_relative_path,
            local_dir="models",
            token=token if token else None # Use token if provided
        )
        print(f"✅ Successfully downloaded {filename_only}\n")
    except Exception as e:
        print(f"❌ Error downloading {filename_only}: {str(e)}\n")

# Build execution list
download_queue = []

if text_encoder_model != "None":
    download_queue.append(f"text_encoders/{text_encoder_model}")

if download_audio_vae:
    download_queue.append("vae/minimax_h3_audio_vae_fp32.safetensors")

if download_video_vae:
    download_queue.append("vae/minimax_h3_video_vae_fp16.safetensors")

if unet_model != "None":
    download_queue.append(f"unet/{unet_model}")

# Execute downloads
print("Initializing High-Speed Download Queue...\n")
token_str = hf_token.strip()

for rel_path in download_queue:
    download_file(rel_path, token_str)

clear_output()

# @title 3. 🚀 Run ComfyUI
use_cloudflare = True # @param {type:"boolean"}
use_interface_in_cell = False # @param {type:"boolean"}

# @markdown ### **VRAM Management**
# @markdown Select how you want ComfyUI to handle model loading:
# @markdown - **High VRAM (Keep loaded):** Keeps models in VRAM for faster subsequent generation.
# @markdown - **Normal VRAM (Load/Unload):** Offloads models from VRAM after use to save memory. (Default)
vram_management = "Normal VRAM (Load/Unload)" # @param ["Normal VRAM (Load/Unload)", "High VRAM (Keep loaded)"]

import torch
import os
from IPython.display import clear_output

clear_output()

%cd /content/ComfyUI

# Configure Launch Args based on selection
launch_args = "--enable-cors-header"

if vram_management == "High VRAM (Keep loaded)":
    launch_args += " --highvram"
# Note: ComfyUI uses "Normal VRAM" behavior by default, so we don't need to add any flags for it.

if use_cloudflare:
    if not os.path.exists("cloudflared-linux-amd64.deb"):
        !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
        !dpkg -i cloudflared-linux-amd64.deb

    import subprocess
    import threading
    import time
    import socket

    def iframe_thread(port):
        while True:
            time.sleep(0.5)
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            result = sock.connect_ex(('127.0.0.1', port))
            if result == 0:
                break
            sock.close()
        print("\nComfyUI finished loading, launching Cloudflare tunnel...\n")

        p = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        for line in p.stderr:
            l = line.decode()
            if "trycloudflare.com " in l:
                print("This is your ComfyUI URL:", l[l.find("http"):], end='')
        clear_output()

    threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

    !python main.py $launch_args

elif use_interface_in_cell:
    import threading
    import time
    import socket

    def iframe_thread(port):
        while True:
            time.sleep(0.5)
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            result = sock.connect_ex(('127.0.0.1', port))
            if result == 0:
                break
            sock.close()
        from google.colab import output
        output.serve_kernel_port_as_iframe(port, height=1024)
        clear_output()
        print("To open in a standalone window click here:")
        output.serve_kernel_port_as_window(port)

    threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

    !python main.py $launch_args

else:
    import socket, time, threading
    from google.colab import output

    def link_thread(port):
        while True:
            time.sleep(0.5)
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            result = sock.connect_ex(('127.0.0.1', port))
            if result == 0:
                break
            sock.close()
        clear_output()
        print("Click the link below to launch the ComfyUI interface:")
        output.serve_kernel_port_as_window(port)

    threading.Thread(target=link_thread, daemon=True, args=(8188,)).start()

    !python main.py $launch_args